# Name That LLM: features from a frozen language model

Generated by `python -m src.build_notebook` from
https://github.com/Dana-Lev/name-that-llm. The cells below recreate the `src/`
package, then extract language-model features for `train.csv` and cross-validate them. This is an experiment: do not submit it.

In [ ]:
from pathlib import Path

for folder in ['src', 'src/features', 'src/models']:
    Path(folder).mkdir(parents=True, exist_ok=True)
for empty_file in ['src/__init__.py', 'src/features/__init__.py']:
    Path(empty_file).touch()

In [ ]:
%%writefile src/data.py
"""Loading the competition data and building the cross-validation folds."""

import os
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

# The 18 valid labels, grouped by lab. Order is fixed: column k of every
# probability matrix in this project refers to LABELS[k].
LABS = {
    "OpenAI": ["gpt-5.5", "gpt-5.6-luna", "gpt-5.6-sol", "gpt-5.6-terra", "gpt-6-astra"],
    "Google": [
        "gemini-3.1-pro-preview",
        "gemini-3.6-flash",
        "gemini-3.7-flash",
        "gemini-3.8-flash",
    ],
    "Anthropic": [
        "claude-fable-5",
        "claude-opus-4-6",
        "claude-opus-4-8",
        "claude-opus-5",
        "claude-sonnet-5",
    ],
    "Qwen": ["qwen3.8-max"],
    "ZAI": ["glm-5.3", "glm-5.3-flash"],
    "Moonshot AI": ["kimi-k3"],
}
LABELS = [model for models in LABS.values() for model in models]
LABEL_TO_ID = {label: i for i, label in enumerate(LABELS)}
LAB_OF = {model: lab for lab, models in LABS.items() for model in models}

N_SPLITS = 5
SEED = 42


def find_data_dir():
    """The folder with train.csv: $NTL_DATA_DIR, Kaggle's input folder, or ./data."""
    candidates = [
        os.environ.get("NTL_DATA_DIR"),
        "/kaggle/input/competitions/name-that-llm",
        "/kaggle/input/name-that-llm",
        "data",
    ]
    for candidate in candidates:
        if candidate and (Path(candidate) / "train.csv").exists():
            return Path(candidate)
    raise FileNotFoundError("train.csv not found; set NTL_DATA_DIR or put the data in ./data")


def load_train(data_dir="data"):
    """Read train.csv and add an integer `label` column."""
    df = pd.read_csv(Path(data_dir) / "train.csv")
    unknown = set(df["model"]) - set(LABELS)
    if unknown:
        raise ValueError(f"train.csv has labels outside the 18 valid ones: {sorted(unknown)}")
    df["label"] = df["model"].map(LABEL_TO_ID)
    return df


def load_test(data_dir="data"):
    return pd.read_csv(Path(data_dir) / "test.csv")


def make_folds(df, n_splits=N_SPLITS, seed=SEED):
    """Assign each row a fold id in [0, n_splits).

    All passages answering the same prompt land in the same fold, so every
    validation prompt is unseen during training, as in the hidden test set.
    Folds are also stratified by model to keep the class mix similar.
    """
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    folds = np.full(len(df), -1, dtype=int)
    for fold, (_, valid_idx) in enumerate(
        splitter.split(df["text"], df["label"], groups=df["prompt_id"])
    ):
        folds[valid_idx] = fold
    return folds


In [ ]:
%%writefile src/evaluate.py
"""Scoring: balanced accuracy (the competition metric) and confusion tables."""

import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score, confusion_matrix

from src.data import LAB_OF, LABELS


def balanced_accuracy(y_true, y_pred):
    """Mean per-class recall over the classes present in y_true."""
    return balanced_accuracy_score(y_true, y_pred)


def fold_scores(y_true, y_pred, folds):
    """Balanced accuracy of each validation fold."""
    return np.array(
        [balanced_accuracy(y_true[folds == f], y_pred[folds == f]) for f in np.unique(folds)]
    )


def model_confusion(y_true, y_pred):
    """18x18 table: rows are the true model, columns the predicted model."""
    ids = np.arange(len(LABELS))
    return pd.DataFrame(confusion_matrix(y_true, y_pred, labels=ids), index=LABELS, columns=LABELS)


def lab_confusion(y_true, y_pred):
    """The same table collapsed to the 6 labs."""
    table = model_confusion(y_true, y_pred)
    # An array, not a list: pandas reads a list of strings as column names.
    labs = np.array([LAB_OF[m] for m in LABELS])
    return table.groupby(labs, sort=False).sum().T.groupby(labs, sort=False).sum().T


def within_lab_error_share(y_true, y_pred):
    """Fraction of the mistakes where the predicted model is from the true lab."""
    table = lab_confusion(y_true, y_pred)
    errors = len(y_true) - int(np.sum(np.asarray(y_true) == np.asarray(y_pred)))
    if errors == 0:
        return 0.0
    model_table = model_confusion(y_true, y_pred).to_numpy()
    same_lab_total = int(np.trace(table.to_numpy()))
    same_lab_errors = same_lab_total - int(np.trace(model_table))
    return same_lab_errors / errors


In [ ]:
%%writefile src/features/tfidf.py
"""TF-IDF features over word and character n-grams."""

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion


def build_tfidf(word_ngrams=(1, 2), char_ngrams=(2, 5), min_df=2):
    """Word n-grams capture vocabulary; character n-grams capture punctuation
    and formatting habits. Case is kept because capitalisation is a style cue."""
    word = TfidfVectorizer(
        analyzer="word",
        ngram_range=word_ngrams,
        min_df=min_df,
        sublinear_tf=True,
        lowercase=False,
        token_pattern=r"(?u)\b\w+\b|[^\w\s]",
    )
    char = TfidfVectorizer(
        analyzer="char",
        ngram_range=char_ngrams,
        min_df=min_df,
        sublinear_tf=True,
        lowercase=False,
    )
    return FeatureUnion([("word", word), ("char", char)])


In [ ]:
%%writefile src/features/stylometry.py
"""Hand-made writing-style features: sentence shape, punctuation habits, formatting.

TF-IDF sees which words and character sequences appear. These features add what
it can't see: how long and how varied the sentences are, how the passage starts
and ends, and how often each punctuation mark is used per 100 characters.
"""

import re

import numpy as np

PUNCTUATION = {
    "comma": ",",
    "semicolon": ";",
    "colon": ":",
    "em_dash": "—",
    "en_dash": "–",
    "hyphen": "-",
    "open_paren": "(",
    "close_paren": ")",
    "straight_double_quote": '"',
    "straight_single_quote": "'",
    "curly_apostrophe": "’",
    "curly_open_quote": "“",
    "question": "?",
    "exclamation": "!",
    "ellipsis_char": "…",
    "three_dots": "...",
    "asterisk": "*",
    "backtick": "`",
    "hash": "#",
    "dollar": "$",
    "backslash": "\\",
    "slash": "/",
    "percent": "%",
    "equals": "=",
    "newline": "\n",
    "blank_line": "\n\n",
    "double_space": "  ",
}

SHAPE_NAMES = [
    "n_chars",
    "n_words",
    "n_sentences",
    "sentence_len_mean",
    "sentence_len_std",
    "sentence_len_max",
    "sentence_len_min",
    "word_len_mean",
    "word_len_std",
    "long_word_share",
    "distinct_word_share",
    "uppercase_share",
    "digit_share",
    "non_ascii_share",
    "starts_uppercase",
    "ends_with_sentence_mark",
    "ends_with_period",
    "pronoun_share",
    "n_list_items",
    "n_bold_spans",
    "n_latex_spans",
]
FEATURE_NAMES = SHAPE_NAMES + [f"{name}_per_100_chars" for name in PUNCTUATION]

_SENTENCE_END = re.compile(r"(?<=[.!?])\s+")
_PRONOUN = re.compile(r"\b(?:I|we|you)\b")
_LIST_ITEM = re.compile(r"^\s*(?:[-*•]|\d+[.)])\s", flags=re.M)
_BOLD = re.compile(r"\*\*[^*]+\*\*")
_LATEX = re.compile(r"\$[^$]+\$")


def style_vector(text):
    """The style features of one passage, in FEATURE_NAMES order."""
    words = text.split()
    sentences = [s for s in _SENTENCE_END.split(text) if s.strip()]
    sentence_lens = np.array([len(s.split()) for s in sentences] or [0])
    word_lens = np.array([len(w) for w in words] or [0])
    n_chars = max(len(text), 1)
    n_words = max(len(words), 1)
    last = text.rstrip()[-1:]

    shape = [
        len(text),
        len(words),
        len(sentences),
        sentence_lens.mean(),
        sentence_lens.std(),
        sentence_lens.max(),
        sentence_lens.min(),
        word_lens.mean(),
        word_lens.std(),
        (word_lens > 8).mean(),
        len({w.lower() for w in words}) / n_words,
        sum(c.isupper() for c in text) / n_chars,
        sum(c.isdigit() for c in text) / n_chars,
        sum(ord(c) > 127 for c in text) / n_chars,
        float(text[:1].isupper()),
        float(last in ".!?" and last != ""),
        float(last == "."),
        len(_PRONOUN.findall(text)) / n_words,
        len(_LIST_ITEM.findall(text)),
        len(_BOLD.findall(text)),
        len(_LATEX.findall(text)),
    ]
    punctuation = [text.count(mark) / n_chars * 100 for mark in PUNCTUATION.values()]
    return shape + punctuation


def style_features(texts):
    """Matrix of style features, one row per passage."""
    return np.array([style_vector(text) for text in texts], dtype=np.float64)


In [ ]:
%%writefile src/features/llm.py
"""Features read out of a frozen open language model (for example Qwen 3).

The model is not trained. Each passage is run through it once and we keep:

- "stats": 14 numbers on how predictable the passage is to the model
  (log-probability of each actual next token, entropy of the model's
  prediction, rank of the actual token among the model's guesses).
- "L<k>": the hidden vectors of layer k averaged over the passage, for four
  layers spread through the network.

This is heavy: run it on a GPU (Kaggle), not on a laptop.
"""

import json
import os
from pathlib import Path

import numpy as np

STAT_NAMES = [
    "logp_mean", "logp_std", "logp_q05", "logp_q25", "logp_q50", "logp_q75", "logp_q95",
    "entropy_mean", "entropy_std", "logp_plus_entropy_mean",
    "top1_share", "top5_share", "top20_share", "log_rank_mean",
]  # fmt: skip

_loaded = {}  # source -> (tokenizer, model, device), so the model loads once


def find_llm(root="/kaggle/input"):
    """Folder of the attached causal language model.

    $NTL_LLM_DIR wins. Otherwise the first folder under `root` whose
    config.json describes a ...ForCausalLM model and that has weights.
    """
    override = os.environ.get("NTL_LLM_DIR")
    if override:
        return override
    for config in sorted(Path(root).rglob("config.json")):
        folder = config.parent
        try:
            architectures = json.loads(config.read_text()).get("architectures") or []
        except (json.JSONDecodeError, OSError):
            continue
        has_weights = any(folder.glob("*.safetensors")) or (folder / "pytorch_model.bin").exists()
        if has_weights and any(a.endswith("ForCausalLM") for a in architectures):
            return str(folder)
    raise FileNotFoundError(f"no causal language model under {root}: attach one (e.g. Qwen 3) as an input")


def _load(source):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    if source not in _loaded:
        device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
        tokenizer = AutoTokenizer.from_pretrained(source)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        tokenizer.padding_side = "right"
        half = torch.float32 if device == "cpu" else torch.float16
        try:  # the argument was renamed from torch_dtype to dtype in newer transformers
            model = AutoModelForCausalLM.from_pretrained(source, dtype=half)
        except TypeError:
            model = AutoModelForCausalLM.from_pretrained(source, torch_dtype=half)
        model = model.to(device).eval()
        _loaded[source] = (tokenizer, model, device)
    return _loaded[source]


def pick_layers(n_layers):
    return sorted({n_layers // 4, n_layers // 2, 3 * n_layers // 4, n_layers})


def extract(texts, source=None, batch_size=8, max_length=512, verbose=True, stats=True):
    """Return {"stats": (n, 14), "L<k>": (n, hidden_size), ...} for the passages.

    stats=False skips the predictability numbers, which are the slow part.
    """
    import torch

    texts = list(texts)
    source = find_llm() if source is None else source
    tokenizer, model, device = _load(source)
    layers = pick_layers(model.config.num_hidden_layers)
    features = {f"L{k}": np.zeros((len(texts), model.config.hidden_size), dtype=np.float32) for k in layers}
    features["stats"] = np.zeros((len(texts), len(STAT_NAMES)), dtype=np.float32)

    order = np.argsort([len(text) for text in texts], kind="stable")
    with torch.no_grad():
        for start in range(0, len(order), batch_size):
            rows = order[start : start + batch_size]
            # An empty passage would give an empty token row; one space keeps it valid.
            batch = [texts[i] if texts[i].strip() else " " for i in rows]
            encoded = tokenizer(
                batch, return_tensors="pt", padding=True, truncation=True, max_length=max_length
            ).to(device)
            out = model(**encoded, output_hidden_states=True)
            mask = encoded["attention_mask"]
            weights = mask.unsqueeze(-1).float()
            for k in layers:
                hidden = out.hidden_states[k].float()
                features[f"L{k}"][rows] = ((hidden * weights).sum(1) / weights.sum(1)).cpu().numpy()

            # Position t predicts token t+1. One passage at a time: the scores
            # over the whole vocabulary (150k words per token) for a full batch
            # don't fit in a 16 GB GPU.
            lengths = mask.sum(1).tolist()
            for j, i in enumerate(rows if stats else []):
                n = int(lengths[j])
                if n < 2:  # a one-token passage has nothing to predict
                    continue
                log_probs = torch.log_softmax(out.logits[j, : n - 1].float(), dim=-1)
                target = encoded["input_ids"][j, 1:n].unsqueeze(-1)
                actual = log_probs.gather(-1, target)
                lp = actual.squeeze(-1).cpu().numpy()
                en = (-(log_probs.exp() * log_probs).sum(-1)).cpu().numpy()
                rk = (log_probs > actual).sum(-1).cpu().numpy()
                del log_probs
                features["stats"][i] = [
                    lp.mean(), lp.std(), *np.quantile(lp, [0.05, 0.25, 0.5, 0.75, 0.95]),
                    en.mean(), en.std(), (lp + en).mean(),
                    (rk == 0).mean(), (rk < 5).mean(), (rk < 20).mean(), np.log1p(rk).mean(),
                ]  # fmt: skip
            if verbose and (start // batch_size) % 50 == 0:
                print(f"llm features: {start}/{len(texts)} passages", flush=True)
    return features


def early_layer_features(texts):
    """Averaged hidden vectors of the layer a quarter of the way up the model.

    In cross-validation this early layer named the model better than the
    middle or last layers (Qwen 3 4B: 0.379 against 0.336 and 0.316).
    """
    features = extract(texts, stats=False, verbose=False)
    first_layer = min((name for name in features if name != "stats"), key=lambda name: int(name[1:]))
    return features[first_layer]


In [ ]:
%%writefile src/models/linear.py
"""Linear models: logistic regression on TF-IDF features and on style features."""

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

from src.features.stylometry import style_features
from src.features.tfidf import build_tfidf


def build_tfidf_logreg(C=10.0, max_iter=3000, **tfidf_kwargs):
    """class_weight="balanced" matches the metric, which weights every model equally."""
    return Pipeline(
        [
            ("tfidf", build_tfidf(**tfidf_kwargs)),
            ("clf", LogisticRegression(C=C, max_iter=max_iter, class_weight="balanced")),
        ]
    )


def build_style_logreg(C=1.0, max_iter=3000):
    """The style features are on very different scales, so standardise them first."""
    return Pipeline(
        [
            ("style", FunctionTransformer(style_features)),
            ("scale", StandardScaler()),
            ("clf", LogisticRegression(C=C, max_iter=max_iter, class_weight="balanced")),
        ]
    )


def build_llm_logreg(C=0.01, max_iter=2000):
    """Logistic regression on features from a frozen language model (src/features/llm.py).

    Needs a GPU and the language model attached; see find_llm.
    """
    from src.features.llm import early_layer_features

    return Pipeline(
        [
            ("features", FunctionTransformer(early_layer_features)),
            ("scale", StandardScaler()),
            ("clf", LogisticRegression(C=C, max_iter=max_iter, class_weight="balanced")),
        ]
    )


In [ ]:
%%writefile src/models/blend.py
"""Combine several models by adding their log-probabilities."""

import numpy as np

EPS = 1e-12


class LogProbBlend:
    """Weighted sum of the members' log-probabilities, renormalised.

    Adding log-probabilities multiplies probabilities, so a class scores high
    only if every member finds it plausible. A weight below 1 makes a member's
    opinion count for less.
    """

    def __init__(self, members):
        """members: list of (build_model, weight) pairs."""
        self.members = members

    def fit(self, texts, y):
        self.models_ = [build().fit(texts, y) for build, _ in self.members]
        self.classes_ = self.models_[0].classes_
        return self

    def predict_proba(self, texts):
        total = sum(
            weight * np.log(model.predict_proba(texts) + EPS)
            for model, (_, weight) in zip(self.models_, self.members)
        )
        total -= total.max(axis=1, keepdims=True)
        probs = np.exp(total)
        return probs / probs.sum(axis=1, keepdims=True)

    def predict(self, texts):
        return self.classes_[self.predict_proba(texts).argmax(axis=1)]


In [ ]:
%%writefile src/models/transformer.py
"""Fine-tune a pretrained text encoder (DeBERTa-v3) to name the model.

The encoder reads the passage as tokens, we average its output vectors over
the passage and put one linear layer with 18 outputs on top. All weights are
trained, with a small learning rate for the pretrained part.

torch and transformers are imported inside the class so the rest of the
project works without them.
"""

import os
from pathlib import Path

import numpy as np

DEFAULT_MODEL = "microsoft/deberta-v3-base"


def find_pretrained(name=DEFAULT_MODEL):
    """Where to load the pretrained weights from.

    Kaggle notebooks run without internet, so there the weights must be
    attached as an input; we look for a folder under /kaggle/input whose path
    contains the model's short name. Elsewhere the Hugging Face name is used
    (downloaded once, then cached). $NTL_PRETRAINED_DIR overrides both.
    """
    override = os.environ.get("NTL_PRETRAINED_DIR")
    if override:
        return override
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        return find_weights_folder(kaggle_input, name)
    return name


def _squash(text):
    """Lower-case letters and digits only, so 'DeBERTa-v3-base' matches 'debertav3base'."""
    return "".join(ch for ch in text.lower() if ch.isalnum())


def find_weights_folder(root, name=DEFAULT_MODEL):
    """The folder under `root` that holds the pretrained model `name`.

    It must have a config.json next to a weights file, and its path must
    contain the model's short name (ignoring case and punctuation). An input
    that is merely called deberta-v3-base but has no weights doesn't count.
    """
    short = name.split("/")[-1]
    for config in sorted(Path(root).rglob("config.json")):
        folder = config.parent
        has_weights = any((folder / f).exists() for f in ("model.safetensors", "pytorch_model.bin"))
        if has_weights and _squash(short) in _squash(str(folder)):
            return str(folder)
    raise FileNotFoundError(
        f"no pretrained {short} weights under {root}: attach an input that contains "
        "config.json together with pytorch_model.bin or model.safetensors"
    )


def pick_device():
    import torch

    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


class FineTunedEncoder:
    """Same interface as the scikit-learn models: fit, predict_proba, classes_."""

    def __init__(
        self,
        pretrained=DEFAULT_MODEL,
        epochs=6,
        lr=3e-5,
        head_lr_factor=20,
        batch_size=16,
        max_length=512,
        label_smoothing=0.05,
        seed=42,
        verbose=False,
    ):
        self.pretrained = pretrained
        self.epochs = epochs
        self.lr = lr
        self.head_lr_factor = head_lr_factor
        self.batch_size = batch_size
        self.max_length = max_length
        self.label_smoothing = label_smoothing
        self.seed = seed
        self.verbose = verbose

    def _batches(self, texts, order):
        import torch

        for start in range(0, len(order), self.batch_size):
            rows = order[start : start + self.batch_size]
            encoded = self.tokenizer_(
                [texts[i] for i in rows],
                truncation=True,
                max_length=self.max_length,
                padding=True,
                return_tensors="pt",
            )
            yield (
                rows,
                encoded["input_ids"].to(self.device_),
                encoded["attention_mask"].to(self.device_),
            )

    def _logits(self, ids, mask):
        hidden = self.encoder_(input_ids=ids, attention_mask=mask).last_hidden_state
        weights = mask.unsqueeze(-1).float()
        # Mean over real tokens only; padding has mask 0.
        pooled = (hidden * weights).sum(1) / weights.sum(1)
        return self.head_(self.dropout_(pooled))

    def fit(self, texts, y):
        import torch
        from torch import nn
        from transformers import AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup

        texts = list(texts)
        y = np.asarray(y)
        self.classes_ = np.unique(y)
        targets = np.searchsorted(self.classes_, y)

        torch.manual_seed(self.seed)
        rng = np.random.default_rng(self.seed)
        self.device_ = pick_device()
        source = find_pretrained(self.pretrained)
        self.tokenizer_ = AutoTokenizer.from_pretrained(source)
        # .float(): the published DeBERTa-v3 weights are stored in 16-bit
        # precision, and training in 16 bits turns the loss into NaN.
        self.encoder_ = AutoModel.from_pretrained(source).float().to(self.device_)
        self.dropout_ = nn.Dropout(0.1)
        self.head_ = nn.Linear(self.encoder_.config.hidden_size, len(self.classes_)).to(self.device_)

        # The new layer starts from random weights, so it gets a larger step
        # than the pretrained encoder.
        optimizer = torch.optim.AdamW(
            [
                {"params": self.encoder_.parameters(), "lr": self.lr},
                {"params": self.head_.parameters(), "lr": self.lr * self.head_lr_factor},
            ],
            weight_decay=0.01,
        )
        steps = self.epochs * int(np.ceil(len(texts) / self.batch_size))
        schedule = get_cosine_schedule_with_warmup(optimizer, int(0.1 * steps), steps)
        parameters = list(self.encoder_.parameters()) + list(self.head_.parameters())

        for epoch in range(self.epochs):
            self.encoder_.train()
            self.dropout_.train()
            total = 0.0
            for rows, ids, mask in self._batches(texts, rng.permutation(len(texts))):
                target = torch.tensor(targets[rows]).to(self.device_)
                loss = nn.functional.cross_entropy(
                    self._logits(ids, mask), target, label_smoothing=self.label_smoothing
                )
                loss.backward()
                nn.utils.clip_grad_norm_(parameters, 1.0)
                optimizer.step()
                schedule.step()
                optimizer.zero_grad()
                total += loss.item() * len(rows)
            if self.verbose:
                print(f"epoch {epoch + 1}/{self.epochs}: training loss {total / len(texts):.3f}", flush=True)
        return self

    def predict_proba(self, texts):
        import torch

        texts = list(texts)
        self.encoder_.eval()
        self.dropout_.eval()
        probs = np.zeros((len(texts), len(self.classes_)))
        # Similar lengths in one batch means less padding, so faster.
        order = np.argsort([len(text) for text in texts], kind="stable")
        with torch.no_grad():
            for rows, ids, mask in self._batches(texts, order):
                probs[rows] = torch.softmax(self._logits(ids, mask), dim=-1).cpu().numpy()
        return probs

    def predict(self, texts):
        return self.classes_[self.predict_proba(texts).argmax(axis=1)]


def build_deberta_base(**kwargs):
    return FineTunedEncoder("microsoft/deberta-v3-base", lr=3e-5, **kwargs)


def build_deberta_small(**kwargs):
    return FineTunedEncoder("microsoft/deberta-v3-small", lr=5e-5, **kwargs)


In [ ]:
%%writefile src/models/__init__.py
"""Every model that can be cross-validated or submitted, by name."""

from src.models.blend import LogProbBlend
from src.models.linear import build_llm_logreg, build_style_logreg, build_tfidf_logreg
from src.models.transformer import build_deberta_base, build_deberta_small


def build_tfidf_style_blend(style_weight=0.5):
    return LogProbBlend([(build_tfidf_logreg, 1.0), (build_style_logreg, style_weight)])


def build_tfidf_style_deberta_blend(style_weight=0.5, deberta_weight=0.5):
    return LogProbBlend(
        [
            (build_tfidf_logreg, 1.0),
            (build_style_logreg, style_weight),
            (build_deberta_base, deberta_weight),
        ]
    )


def build_full_blend(style_weight=0.5, deberta_weight=0.5, llm_weight=0.5):
    return LogProbBlend(
        [
            (build_tfidf_logreg, 1.0),
            (build_style_logreg, style_weight),
            (build_deberta_base, deberta_weight),
            (build_llm_logreg, llm_weight),
        ]
    )


MODELS = {
    "tfidf_logreg": build_tfidf_logreg,
    "style_logreg": build_style_logreg,
    "tfidf_style_blend": build_tfidf_style_blend,
    "deberta_small": build_deberta_small,
    "deberta_base": build_deberta_base,
    "tfidf_style_deberta_blend": build_tfidf_style_deberta_blend,
    "llm_logreg": build_llm_logreg,
    "full_blend": build_full_blend,
}

# Models that fine-tune a pretrained network: minutes per fit, need torch.
NEURAL = {"deberta_small", "deberta_base", "tfidf_style_deberta_blend", "llm_logreg", "full_blend"}


In [ ]:
%%writefile src/prompts.py
"""Using the structure of the data: every model answers each prompt at most once.

In train.csv no prompt has two passages from the same model. If the test set
is built the same way, the predictions for one prompt should all be different
models. The test set has no prompt_id, so we first group passages by topic
(passages answering one prompt share vocabulary), then pick the best labelling
of each group in which no model repeats.

Whether the hidden test set follows this structure is an assumption; the
leaderboard is the only way to check it.
"""

import numpy as np
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import AgglomerativeClustering
from sklearn.feature_extraction.text import TfidfVectorizer

EPS = 1e-12


def cluster_prompts(texts, threshold=0.90):
    """Group passages that seem to answer the same prompt; returns a group id per passage.

    Average-linkage clustering on cosine distance between TF-IDF vectors with
    stop words removed, so only topic words count. Merging stops at `threshold`,
    so the number of prompts doesn't have to be known. 0.90 was chosen on
    train.csv, where the true prompts are known.
    """
    texts = list(texts)
    if len(texts) < 2:
        return np.zeros(len(texts), dtype=int)
    vectorizer = TfidfVectorizer(sublinear_tf=True, stop_words="english", min_df=1)
    vectors = vectorizer.fit_transform(texts).toarray()
    # A passage with no topic words has a zero vector, which has no cosine distance.
    empty = ~vectors.any(axis=1)
    vectors[empty, 0] = 1.0
    clustering = AgglomerativeClustering(
        n_clusters=None, distance_threshold=threshold, metric="cosine", linkage="average"
    )
    return clustering.fit_predict(vectors)


def assign_within_groups(probs, groups):
    """Label each group so that no class repeats, maximising total log-probability.

    This is the assignment problem, solved exactly by the Hungarian algorithm.
    A group larger than the number of classes can't be a single prompt (the
    clustering merged several), so there each class may be used once per
    merged prompt: ceil(group size / number of classes) times.
    """
    probs = np.asarray(probs)
    groups = np.asarray(groups)
    n_classes = probs.shape[1]
    log_probs = np.log(probs + EPS)
    predicted = np.zeros(len(probs), dtype=int)
    for group in np.unique(groups):
        rows = np.where(groups == group)[0]
        copies = int(np.ceil(len(rows) / n_classes))
        row_idx, col_idx = linear_sum_assignment(-np.tile(log_probs[rows], (1, copies)))
        predicted[rows[row_idx]] = col_idx % n_classes
    return predicted


In [ ]:
%%writefile src/predict.py
"""Fit a model on all of train.csv and write submission.parquet.

    python -m src.predict --model tfidf_style_blend --data-dir data

This is what the Kaggle submission notebooks run.
"""

import argparse

import pandas as pd

from src.data import LABELS, find_data_dir, load_test, load_train
from src.models import MODELS
from src.prompts import assign_within_groups, cluster_prompts


def check_submission(submission, test):
    """Fail here rather than have Kaggle reject the file."""
    if list(submission.columns) != ["id", "model"]:
        raise ValueError(f"columns must be id, model; got {list(submission.columns)}")
    if submission.isna().any().any():
        raise ValueError("submission has null values")
    if submission["id"].duplicated().any() or set(submission["id"]) != set(test["id"]):
        raise ValueError("every test id must appear exactly once")
    unknown = set(submission["model"]) - set(LABELS)
    if unknown:
        raise ValueError(f"labels outside the 18 valid ones: {sorted(unknown)}")


def make_submission(model="tfidf_logreg", assign=False, data_dir=None, out="submission.parquet"):
    """With assign=True, group the test passages by prompt and let no model
    repeat inside a group (see src/prompts.py)."""
    data_dir = find_data_dir() if data_dir is None else data_dir
    train, test = load_train(data_dir), load_test(data_dir)

    fitted = MODELS[model]().fit(train["text"], train["label"])
    # Columns of predict_proba follow fitted.classes_, which are label ids.
    probs = fitted.predict_proba(test["text"])
    if assign:
        groups = cluster_prompts(test["text"])
        columns = assign_within_groups(probs, groups)
        print(f"grouped {len(test)} passages into {groups.max() + 1} prompts")
    else:
        columns = probs.argmax(axis=1)
    predicted = fitted.classes_[columns]

    submission = pd.DataFrame({"id": test["id"], "model": [LABELS[i] for i in predicted]})
    check_submission(submission, test)
    submission.to_parquet(out, index=False)
    print(f"trained on {len(train)} passages, wrote {len(submission)} predictions to {out}")
    return submission


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    parser.add_argument("--model", default="tfidf_logreg", choices=sorted(MODELS))
    parser.add_argument("--assign", action="store_true")
    parser.add_argument("--data-dir", default=None)
    parser.add_argument("--out", default="submission.parquet")
    args = parser.parse_args()
    make_submission(args.model, args.assign, args.data_dir, args.out)


In [ ]:
%%writefile src/llm_experiment.py
"""Experiment: how well do frozen-LLM features name the model?

Run on Kaggle with a GPU and an open language model attached:

    from src.llm_experiment import main; main()

It extracts the features of every training passage once, then cross-validates
a logistic regression on each block of features with the project's folds.
Everything lands in the output folder: the features (llm_features.npz), the
out-of-fold probabilities of each variant (oof_llm_<block>.npy) and a table
(llm_experiment.csv). The oof files go into oof/ locally for blending with
src.blend_oof.
"""

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from src.data import LABELS, find_data_dir, load_train, make_folds
from src.evaluate import fold_scores
from src.features.llm import extract, find_llm

C_VALUES = (0.01, 0.1, 1.0)


def cross_validate_block(X, y, folds, C):
    oof = np.zeros((len(y), len(LABELS)))
    for fold in np.unique(folds):
        train, valid = folds != fold, folds == fold
        model = make_pipeline(
            StandardScaler(), LogisticRegression(C=C, max_iter=2000, class_weight="balanced")
        )
        model.fit(X[train], y[train])
        oof[np.ix_(valid, model.classes_)] = model.predict_proba(X[valid])
    return oof


def main(out_dir="."):
    out_dir = Path(out_dir)
    df = load_train(find_data_dir())
    folds = make_folds(df)
    y = df["label"].to_numpy()
    source = find_llm()
    print(f"language model: {source}")

    features = extract(df["text"], source)
    np.savez_compressed(out_dir / "llm_features.npz", **features)

    layer_names = [name for name in features if name != "stats"]
    blocks = dict(features)
    blocks["all_layers"] = np.hstack([features[name] for name in layer_names])
    blocks["all_layers+stats"] = np.hstack([blocks["all_layers"], features["stats"]])

    rows = []
    for name, X in blocks.items():
        best = None
        for C in C_VALUES:
            oof = cross_validate_block(X, y, folds, C)
            scores = fold_scores(y, oof.argmax(axis=1), folds)
            rows.append({"features": name, "n_features": X.shape[1], "C": C,
                         "cv_mean": scores.mean(), "cv_std": scores.std()})  # fmt: skip
            print(f"{name:18s} C={C:<5} {scores.mean():.4f} +- {scores.std():.4f}", flush=True)
            if best is None or scores.mean() > best[0]:
                best = (scores.mean(), oof)
        np.save(out_dir / f"oof_llm_{name.replace('+', '_')}.npy", best[1])

    table = pd.DataFrame(rows)
    table.to_csv(out_dir / "llm_experiment.csv", index=False)
    return table


In [ ]:
from src.llm_experiment import main

table = main()
table.sort_values("cv_mean", ascending=False)